## Project Setup with all the relevant functions

In [ ]:
#import necessary packages

from statsbombpy import sb
import mplsoccer
from mplsoccer import Pitch
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from networkx.algorithms.community import kernighan_lin_bisection
from networkx.algorithms.community import modularity
from networkx.algorithms.community import girvan_newman
import random
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from networkx.algorithms.triads import triadic_census
from collections import defaultdict
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.manifold import TSNE
import os
os.environ['KERAS_BACKEND']='tensorflow'
import keras
import tensorflow as tf
from keras.layers import Layer, Dense, Reshape, Dropout
from keras.models import Model
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from scipy.spatial.distance import cdist

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from pathlib import Path

from collections import Counter

from scipy.stats import spearmanr,pearsonr

##### Graph creation Player Based Graph 

In [66]:
def create_passes_db (events, team_id):
    
    passes = events[
    (events['team_id'] == team_id) &
    (events['type'] == 'Pass') &
    (events['pass_outcome'].isna()) & # successful passes only
    (events['location'].notna()) # location must be not NaN
    ]

    valid_players = passes['player'].dropna().unique() # list of all the active players

    passes = passes[passes['pass_recipient'].isin(valid_players)] # remove any players that received only passes

    return passes

In [67]:
def G_dist_gen(passes):

    #creation of the edge list

    edge_list = (
        passes
        .groupby(['player', 'pass_recipient'])
        .size()
        .reset_index(name='pass_freq')
    )

    # creation of distance as inverse of the pass frequencies. it will be used as weight in the network
    edge_list['distance'] = 1/edge_list['pass_freq']

    #creation of the graph

    G_dist = nx.DiGraph()

    for _, row in edge_list.iterrows():
        G_dist.add_edge(
            row['player'],
            row['pass_recipient'],
            weight=row['distance'],
            pass_freq=row["pass_freq"]
        )

    return G_dist

In [68]:
def player_position(passes):
    
    passer_pos = (
        passes
        .groupby("player")["location"]
        .apply(lambda x: pd.DataFrame(x.tolist()).mean())
        .unstack()
    )

    passer_pos.columns = ["x_pass", "y_pass"]


    # Average position when the player receives the ball
    receiver_pos = (
        passes
        .groupby("pass_recipient")["pass_end_location"]
        .apply(lambda x: pd.DataFrame(x.tolist()).mean())
        .unstack()
    )

    receiver_pos.columns = ["x_receive", "y_receive"]


    # Combine passing and receiving positions
    player_pos = passer_pos.join(
        receiver_pos,
        how="outer"
    )

    # Final average position
    player_pos["x"] = player_pos[["x_pass", "x_receive"]].mean(axis=1)
    player_pos["y"] = player_pos[["y_pass", "y_receive"]].mean(axis=1)

    # Keep only final coordinates
    player_pos = player_pos[["x", "y"]]


    # Create position dictionary for NetworkX
    pos = {
        player: (row.x, row.y)
        for player, row in player_pos.iterrows()
    }

    return pos

##### Graph creation Positional Based Graph 

In [69]:
def get_pitch_section(location):
    """
    Assign a StatsBomb location [x, y] to one of the 6 pitch sections.
    Pitch size: 120 x 80
    """
    if location is None or pd.isna(location).any():
        return np.nan

    x, y = location[0], location[1]

    if 0 <= x < 40 and 0 <= y < 20:
        return "Section 1"
    elif 40 <= x < 80 and 0 <= y < 20:
        return "Section 2"
    elif 80 <= x <= 120 and 0 <= y < 20:
        return "Section 3"
    elif 0 <= x < 40 and 20 <= y <= 40:
        return "Section 4"
    elif 40 <= x < 80 and 20 <= y <= 40:
        return "Section 5"
    elif 80 <= x <= 120 and 20 <= y <= 40:
        return "Section 6"
    elif 0 <= x < 40 and 40 <= y < 60:
        return "Section 7"
    elif 40 <= x < 80 and 40 <= y < 60:
        return "Section 8"
    elif 80 <= x <= 120 and 40 <= y < 60:
        return "Section 9"
    elif 0 <= x < 40 and 60 <= y <= 80:
        return "Section 10"
    elif 40 <= x < 80 and 60 <= y <= 80:
        return "Section 11"
    elif 80 <= x <= 120 and 60 <= y <= 80:
        return "Section 12"
    else:
        return np.nan


In [70]:
def G_section_gen(passes):

    passes["area_player"] = passes["location"].apply(get_pitch_section)
    passes["area_pass_receiver"] = passes["pass_end_location"].apply(get_pitch_section)

    # Keep only valid positional passes
    positional_passes = passes.dropna(subset=["area_player", "area_pass_receiver"])

    # Create edge list between pitch sections
    section_edge_list = (
        positional_passes
        .groupby(["area_player", "area_pass_receiver"])
        .size()
        .reset_index(name="pass_freq")
    )

    # Add distance as inverse of frequency
    section_edge_list["distance"] = 1 / section_edge_list["pass_freq"]

    G_sections = nx.DiGraph()

    for _, row in section_edge_list.iterrows():
        G_sections.add_edge(
            row["area_player"],
            row["area_pass_receiver"],
            weight=row["distance"],
            pass_freq=row["pass_freq"]
        )
    
    return G_sections

In [71]:
def section_position(passes):

    passes["area_player"] = passes["location"].apply(get_pitch_section)
    passes["area_pass_receiver"] = passes["pass_end_location"].apply(get_pitch_section)
  
    section_pos = (
        passes
        .groupby('area_player')['location']
        .apply(lambda x: pd.DataFrame(x.tolist()).mean())
        .unstack()
    )

    section_pos.columns = ['x', 'y']

    pos_section = {
        section: (row.x, row.y)
        for section, row in section_pos.iterrows()
    }

    return pos_section

### Key Graph-Theoretical Features

In [72]:
def graph_level_metrics(G):
    """
    Compute graph-level metrics and normalized triad frequencies
    for a football passing network.

    Nodes = players
    Edges = passes
    weight = distance = 1 / pass frequency
    """

    metrics = {}

    n = G.number_of_nodes()
    m = G.number_of_edges()

    metrics["number_of_players"] = n
    metrics["number_of_edges"] = m

    if n == 0 or m == 0:
        return metrics

    #0. Num. Passes

    num_pass=0

    for u, v in G.edges():
        num_pass = num_pass + G[u][v]["pass_freq"]

    metrics['num_passes'] = num_pass

    # 1. Density
    metrics["density"] = nx.density(G)

    # 2. Average degree
    degrees = dict(G.degree())
    metrics["average_degree"] = np.mean(list(degrees.values())) #move to undirect graph?

    # 3. Degree centralization
    max_degree = max(degrees.values())

    if n > 2:
        metrics["degree_centralization"] = (
            sum(max_degree - k for k in degrees.values()) /
            ((n - 1) * (n - 2))
        )
    else:
        metrics["degree_centralization"] = np.nan

    # Undirected version for clustering, modularity, algebraic connectivity
    G_undirected = G.to_undirected()

    # 4. Clustering coefficient
    metrics["average_clustering"] = nx.average_clustering(
        G_undirected,
        weight=None
    )

    # 5. Modularity
    communities = nx.community.greedy_modularity_communities(
        G_undirected,
        weight="weight"
    )

    metrics["modularity"] = nx.community.modularity(
        G_undirected,
        communities,
        weight="weight"
    )

    metrics["number_of_communities"] = len(communities)

    # 6. Average shortest path length
    if nx.is_strongly_connected(G):
        metrics["average_path_length"] = 1 / nx.average_shortest_path_length(
            G,
            weight="weight"
        )
    else:
        largest_scc = max(nx.strongly_connected_components(G), key=len)
        G_scc = G.subgraph(largest_scc).copy()

        if G_scc.number_of_nodes() > 1:
            metrics["average_path_length"] = 1 / nx.average_shortest_path_length(
                G_scc,
                weight="weight"
            )
        else:
            metrics["average_path_length"] = np.nan

    # 7. Betweenness centrality standard deviation
    betweenness = nx.betweenness_centrality(
        G,
        weight="weight",
        normalized=True
    )

    metrics["betweenness_sd"] = np.std(list(betweenness.values()))

    # 8. Algebraic connectivity
    if nx.is_connected(G_undirected):
        metrics["algebraic_connectivity"] = nx.algebraic_connectivity(
            G_undirected,
            weight="weight"
        )
    else:
        largest_cc = max(nx.connected_components(G_undirected), key=len)
        G_cc = G_undirected.subgraph(largest_cc).copy()

        if G_cc.number_of_nodes() > 1:
            metrics["algebraic_connectivity"] = nx.algebraic_connectivity(
                G_cc,
                weight="weight"
            )
        else:
            metrics["algebraic_connectivity"] = np.nan

    # 9. Degree assortativity
    metrics["degree_assortativity"] = nx.degree_assortativity_coefficient(
        G_undirected
    )

    # 10. Triad census
    triad_freq = triadic_census(G)

    total_triads = sum(triad_freq.values())

    for triad_type, freq in triad_freq.items():

        if total_triads > 0:
            metrics[f"triad_{triad_type}_norm"] = freq / total_triads
        else:
            metrics[f"triad_{triad_type}_norm"] = 0

    # 11. Motif entropy
    triad_probs = np.array([
        freq / total_triads
        for freq in triad_freq.values()
        if freq > 0
    ])

    metrics["entropy_motif"] = -np.sum(
        triad_probs * np.log(triad_probs)
    )

    return metrics

### Stochastic Block Model

In [ ]:
def most_central_edge(G):

    """
    Identify the most 'important' edge in the graph based on edge betweenness centrality.

    Return the parameter for the Girvan–Newman Algorithm.
    """
    centrality = nx.edge_betweenness_centrality(G, weight="weight")

    return max(centrality, key=centrality.get)



def find_community_detection(G):
    """
    Find the community detection based on the Girvan–Newman Algorithm.

    Return the community that has the highest modularity index.
    """
    # Containers to store all tested community partitions and their modularity scores
    partitions = []
    modularity_scores = []

    # Apply the Girvan–Newman community detection algorithm
    for communities in nx.community.girvan_newman(G, most_valuable_edge=most_central_edge):
        communities = tuple(sorted(c) for c in communities)
        
        #store the set of communities
        partitions.append(communities)

        # Compute modularity score for the current partition and store it
        Q = modularity(G, communities, weight='weight')
        modularity_scores.append(Q)

        # Stop early to avoid excessive fragmentation.
        if len(communities) >= 6:
            break


    #extract the partition with the highest Modularity score
    best_idx = np.argmax(modularity_scores)
    best_com = partitions[best_idx]
    best_Q = modularity_scores[best_idx]

    return best_com


In [74]:
def dc_sbm_log_likelihood(G, partition, weight="pass_freq"):
    """
    Degree-corrected SBM log-likelihood.

    G: NetworkX graph
    partition: dict {node: block}
    weight: edge attribute used as edge weight
    """

    blocks = sorted(set(partition.values()))

    # total degree/stub count per block
    kappa = {r: 0 for r in blocks}

    for node in G.nodes():
        r = partition[node]
        kappa[r] += G.degree(node, weight=weight)

    # edge weight between blocks
    m = {(r, s): 0 for r in blocks for s in blocks}

    for u, v, data in G.edges(data=True):
        r = partition[u]
        s = partition[v]
        w = data.get(weight, 1)

        m[(r, s)] += w

        if not G.is_directed():
            m[(s, r)] += w

    # log-likelihood
    L = 0

    for r in blocks:
        for s in blocks:
            m_rs = m[(r, s)]

            if m_rs > 0 and kappa[r] > 0 and kappa[s] > 0:
                L += m_rs * np.log(m_rs / (kappa[r] * kappa[s]))

    return L

In [75]:
def partition_from_communities(communities):
    """
    Convert community list/tuple into dictionary format:
    {player: block_id}
    """

    partition = {}

    for block_id, community in enumerate(communities):
        for player in community:
            partition[player] = block_id

    return partition

In [76]:
def fit_dc_sbm_from_partition(
    G,
    weight="pass_freq",
    max_iter=100,
    random_state=42):

    random.seed(random_state)
    np.random.seed(random_state)

    initial_partition = find_community_detection(G)

    initial_partition = partition_from_communities(initial_partition)

    partition = initial_partition.copy()
    K = len(set(partition.values()))

    current_L = dc_sbm_log_likelihood(G, partition, weight=weight)

    for iteration in range(max_iter):

        improved = False
        nodes = list(G.nodes())
        random.shuffle(nodes)

        for node in nodes:

            current_block = partition[node]
            best_block = current_block
            best_L = current_L

            for new_block in range(K):

                if new_block == current_block:
                    continue

                test_partition = partition.copy()
                test_partition[node] = new_block

                test_L = dc_sbm_log_likelihood(
                    G,
                    test_partition,
                    weight=weight
                )

                if test_L > best_L:
                    best_L = test_L
                    best_block = new_block

            if best_block != current_block:
                partition[node] = best_block
                current_L = best_L
                improved = True

        if not improved:
            break

    return partition, K

In [77]:
def dc_sbm_gn_results(G):

    dc_partition_gn, K = fit_dc_sbm_from_partition(G)

    dc_sbm_gn_results = pd.DataFrame({
        "player": list(dc_partition_gn.keys()),
        "block": list(dc_partition_gn.values())
    }).sort_values("block")

    dc_partition = []

    for i in range(K):
        dc_partition.append(list(dc_sbm_gn_results[dc_sbm_gn_results['block'] ==i]['player'].values))
    
    return dc_sbm_gn_results, dc_partition

#### Visualisation

In [78]:
def draw_spring(G, pos, com, label):
    """
    Visualize the communities detected in a graph using a spring layout.

    Parameters
    ----------
    G : networkx.Graph
        The input graph.
    com : list of lists
        A list of communities, where each sublist contains the nodes
        belonging to one community.
    """

    # Get the list of node IDs.
    NodeId = list(G.nodes())

    # Define node sizes proportional to their degree (number of connections).
    node_size = [G.degree(i)**1.1 * 60 for i in NodeId]

    # Create a new figure with a specific size.

    pitch = Pitch(
        pitch_type="statsbomb",
        pitch_color="white",
        line_color="black"
    )


    fig, ax = pitch.draw(figsize=(14, 12))

    # Draw the base graph with white nodes and black edges.
    nx.draw(
        G, pos,
        with_labels=True,
        node_size=node_size,
        node_shape='o',
        width=[(1/G[u][v]['weight'])/15 for u, v in G.edges()],
    )

    # Define a list of colors to use for different communities.
    color_list = ['pink', 'orange', 'r', 'g', 'b', 'y', 'm', 'gray', 'black', 'c', 'brown']

    # Draw each community with a distinct color.
    for i in range(len(com)):
        nx.draw_networkx_nodes(G, pos, nodelist=com[i], node_color=color_list[i])

    plt.title(
        label=label,
        fontsize=14,
        fontweight="bold"
    )

    # Display the plot.
    plt.show()


#### Triads / Motifs

In [79]:
def compute_triad_census(G):
    triad_freq = triadic_census(G)

    # Convert to dataframe
    triad_df = (
        pd.DataFrame.from_dict(
            triad_freq,
            orient='index',
            columns=['frequency']
        )
        .sort_values(by='frequency', ascending=False)
    )

    # Normalize frequencies
    triad_df['normalized_frequency'] = (
        triad_df['frequency'] /
        triad_df['frequency'].sum()
    )

    return triad_df['normalized_frequency'] 


### T-SNE & kMean Scatter Plot representation

In [80]:
def plot_tsne_kMean_graph_embedded(G_embedded,label="Graph Embedded || t-SNE projection with kMean cluster"):

    plt.figure(figsize=(11, 8))

    scatter = sns.scatterplot(
        data=G_embedded,
        x="tsne_1",
        y="tsne_2",
        hue="kMean_cluster",
        size="num_passes",
        sizes=(80, 450),
        palette="Set2",
        alpha=0.8,
        edgecolor="black"
    )

    # Add xG as text annotation
    for _, row in G_embedded.iterrows():
        plt.text(
            row["tsne_1"] + 0.3,
            row["tsne_2"] + 0.3,
            f"C{row['kMean_cluster']} | xG={row['xg_team_match']:.2f}",
            fontsize=8,
            alpha=0.75
        )

    plt.title(
        label=label,
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel("t-SNE component 1")
    plt.ylabel("t-SNE component 2")

    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

### Features Embeddings

In [81]:
def build_features_graph(matches,G_gen):

    all_metrics = []

    for match_id in matches["match_id"].values:

        competition = matches[(matches['match_id'] == match_id)]['competition']
        score = matches[(matches['match_id'] == match_id)]['home_score'] # capture the home score
        home_dummy = 1
        competition_gender = matches[(matches['match_id'] == match_id)]['competition_gender']

        events = sb.events(match_id=match_id)
        
        for team_id in events['team_id'].unique():

            passes = create_passes_db(events, team_id)
            G_match = G_gen(passes)
            metrics = pd.DataFrame([graph_level_metrics(G_match)])
            metrics['xg_team_match'] = events[(events['team_id'] == team_id)]['shot_statsbomb_xg'].dropna().sum()
            metrics["match_id"] = match_id
            metrics["team_id"] = team_id
            metrics["team_name"] = passes["team"].unique()
            metrics["competition"] = competition.values[0]
            metrics['score'] = score.values[0]
            metrics['home_dummy'] = home_dummy
            metrics['competition_gender'] = competition_gender.values[0]
            score = matches[(matches['match_id'] == match_id)]['away_score'] # to capture the away score
            home_dummy = 0
            all_metrics.append(metrics)

    metrics_df = pd.concat(all_metrics, ignore_index=True)

    return metrics_df

In [82]:
def embedded_graphs_prep(embedded_graphs):

    feature_cols = embedded_graphs.select_dtypes(include=[np.number]).columns.tolist()

    # Correct version: remove identifier / target columns
    feature_cols = [
        col for col in feature_cols
        if col not in ["match_id", "team_id", "xg_team_match","score","home_dummy"]
    ]

    X = embedded_graphs[feature_cols]

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, feature_cols

In [83]:
def clustering_graph(X_scaled, K, random_state=100):

    kmeans = KMeans(
        n_clusters=K,
        random_state=random_state,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    return labels, kmeans


In [84]:
def tsne_features(X_scaled, random_state=42):

    perplexity_value = min(10, len(X_scaled) - 1)

    tsne = TSNE(
        n_components=2,
        learning_rate="auto",
        init="pca",
        perplexity=perplexity_value,
        random_state=random_state
    )

    return tsne.fit_transform(X_scaled)

### Graph2Vec

In [ ]:
def weisfeiler_lehman_step(G, node_labels):

    """
    Perform one Weisfeiler-Lehman relabelling step.
    
    Parameters
    ----------
    G : networkx.Graph
        Undirected graph for one football passing network.

    node_labels : dict
        Dictionary mapping each node to its current label.

    Returns
    -------
    new_node_labels : dict
        Updated node labels after one WL iteration.
    """

    new_node_labels = {}

    for node in G.nodes():

        # Collect labels of all neighbouring nodes
        neighbour_labels = [
            node_labels[neighbour]
            for neighbour in G.neighbors(node)
        ]

        # Sort labels so that the representation is permutation-invariant
        neighbour_labels = sorted(neighbour_labels)

        # Combine current node label with neighbourhood labels
        combined_label = (
            str(node_labels[node])
            + "_"
            + "_".join(neighbour_labels)
        )

        new_node_labels[node] = combined_label

    return new_node_labels

In [86]:
def extract_wl_features(G, iterations=2):
    """
    Extract Weisfeiler-Lehman rooted subgraph labels from a graph.

    Parameters
    ----------
    G : networkx.Graph
        Undirected passing graph for one match.

    iterations : int
        Number of WL relabelling iterations.

    Returns
    -------
    wl_words : list of str
    Returns a list of labels that will be treated as words
    in the Graph2Vec-style Doc2Vec model.
    """

    # Initial node labels = pass frequency of the nodes
    node_labels  = {
        node: str(int(G.degree(node, weight="pass_freq"))) # change this with the a list of numer (0,max_node) // name of the player // both of them will be unique (is it what we want)??
        for node in G.nodes()                                                                   
    }

    wl_words  = []

    for _ in range(iterations):

        node_labels = weisfeiler_lehman_step(G, node_labels)

        wl_words.extend(node_labels.values())

    return wl_words

In [87]:
def build_graph2vec_documents(matches, G_gen, iterations=2):
    """
    Build Graph2Vec-style documents for all matches.

    Each match becomes one TaggedDocument.
    """
    documents = []

    for match_id in matches["match_id"].values:

        events = sb.events(match_id=match_id)
                
        for team_id in events['team_id'].unique():
            #generate the graph
            passes = create_passes_db(events, team_id)
            G_match = G_gen(passes)

            # Skip empty or very small graphs
            if G_match.number_of_nodes() < 3 or G_match.number_of_edges() < 2:
                continue
            
            # Convert directed graph to undirected for WL rooted-subgraph extraction
            G_undirected = G_match.to_undirected()

            wl_words = extract_wl_features(G_undirected,iterations=iterations)

            #append the graph tagged as match_id
            documents.append(TaggedDocument(words=wl_words,tags=[str(match_id*10000 + team_id)])) # tags=[str(match_id*10000 + team_id)] this will ensure the uniqueness of the tag || to check the max(team_id)
            #documents.append(TaggedDocument(words=wl_words,tags=[str(match_id)]))

    return documents

### GATE

In [ ]:
def graph_to_matrices(G, passes, max_nodes=None):
    """
    Convert a NetworkX passing graph into:
    - X: node feature matrix
    - A: adjacency matrix
    - node_names: list of players

    Assumption:
    G is a directed weighted graph where edges represent passes.
    """

    node_names = list(G.nodes())
    n_nodes = len(node_names)

    if max_nodes is None:
        max_nodes = n_nodes

    node_index = {
        node: i
        for i, node in enumerate(node_names)
    }
    # Adjacency matrix
    A = np.zeros((max_nodes, max_nodes), dtype=np.float32)

    for u, v, data in G.edges(data=True):

        i = node_index[u]
        j = node_index[v]

        pass_freq = data.get("pass_freq", 1)

        A[i, j] = pass_freq

    # Normalize adjacency to [0, 1]
    if A.max() > 0:
        A = A / A.max()


    # Node features

    pos = player_position(passes)
    in_degree = dict(G.in_degree(weight="pass_freq"))
    out_degree = dict(G.out_degree(weight="pass_freq"))
    total_degree = dict(G.degree(weight="pass_freq"))
    betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
    

    X = np.zeros((max_nodes, 6), dtype=np.float32)

    for node, i in node_index.items():

        x_pos, y_pos = pos.get(node, (np.nan, np.nan))


        X[i, 0] = in_degree.get(node, 0)
        X[i, 1] = out_degree.get(node, 0)
        X[i, 2] = total_degree.get(node, 0)
        X[i, 3] = betweenness.get(node, 0)
        X[i,4] =  x_pos / 120 if not np.isnan(x_pos) else 0
        X[i,5] =  y_pos / 80 if not np.isnan(y_pos) else 0

    return X, A, node_names

In [ ]:
def build_graph_dataset(matches,G_gen):
    """
    Build X and A matrices for all matches.

    Output:
    X_all: shape (num_matches, max_nodes, num_features)
    A_all: shape (num_matches, max_nodes, max_nodes)
    """

    graphs = []
    passes_list = []
    # match_team_list = []

    for match_id in matches["match_id"].values:

        events = sb.events(match_id=match_id)
                
        for team_id in events['team_id'].unique():
            #generate the graph
            passes = create_passes_db(events, team_id)
            G_match = G_gen(passes)

            # Skip empty or very small graphs
            if G_match.number_of_nodes() < 3 or G_match.number_of_edges() < 2:
                continue

            graphs.append(G_match)
            passes_list.append(passes)
            # match_team_list.append([match_id,team_id])

    max_nodes = max(G.number_of_nodes() for G in graphs)

    X_list = []
    A_list = []

    for G, passes in zip(graphs, passes_list):
        X, A, _ = graph_to_matrices(G, passes, max_nodes=max_nodes)
        X_list.append(X)
        A_list.append(A)

    X_all = np.array(X_list, dtype=np.float32)
    A_all = np.array(A_list, dtype=np.float32)

    return X_all, A_all, max_nodes

In [90]:
def scale_node_features(X_all):
    """
    Standardize node features across all matches.

    Input:
    X_all shape = (num_matches, max_nodes, num_features)

    Output:
    X_scaled with same shape.
    """

    num_matches, max_nodes, num_features = X_all.shape

    X_reshaped = X_all.reshape(-1, num_features)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_reshaped)

    X_scaled = X_scaled.reshape(num_matches, max_nodes, num_features)

    return X_scaled.astype(np.float32), scaler

In [91]:
class GraphAttentionLayer(Layer):
    """
    Single-head Graph Attention Layer.

    Input:
    X: node features, shape (batch, n_nodes, input_dim)
    A: adjacency matrix, shape (batch, n_nodes, n_nodes)

    Output:
    H: updated node embeddings, shape (batch, n_nodes, output_dim)
    """

    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.output_dim = output_dim

    def build(self, input_shape):

        input_dim = input_shape[0][-1]

        self.W = self.add_weight(
            shape=(input_dim, self.output_dim),
            initializer="glorot_uniform",
            trainable=True,
            name="W"
        )

        self.attention_vector = self.add_weight(
            shape=(2 * self.output_dim, 1),
            initializer="glorot_uniform",
            trainable=True,
            name="attention_vector"
        )

    def call(self, inputs):

        X, A = inputs

        # Linear transformation of node features
        H = tf.matmul(X, self.W)

        n_nodes = tf.shape(H)[1]

        # Create all pair combinations: h_i and h_j
        H_i = tf.tile(tf.expand_dims(H, axis=2), [1, 1, n_nodes, 1])
        H_j = tf.tile(tf.expand_dims(H, axis=1), [1, n_nodes, 1, 1])

        attention_input = tf.concat([H_i, H_j], axis=-1)

        # Raw attention scores
        e = tf.squeeze(
            tf.matmul(attention_input, self.attention_vector),
            axis=-1
        )

        e = tf.nn.leaky_relu(e)

        # Mask non-existing edges
        negative_infinity = -1e9
        attention_scores = tf.where(
            A > 0,
            e,
            negative_infinity
        )

        # Attention coefficients
        alpha = tf.nn.softmax(attention_scores, axis=-1)

        # Weighted aggregation of neighbours
        output = tf.matmul(alpha, H)

        return tf.nn.elu(output)

In [92]:
class GATEncoder(Layer):
    """
    Encoder that transforms node features into node embeddings.
    """

    def __init__(self, hidden_dim=64, embedding_dim=128):
        super().__init__()

        self.gat_1 = GraphAttentionLayer(hidden_dim)
        self.gat_2 = GraphAttentionLayer(embedding_dim)

    def call(self, inputs):

        X, A = inputs

        H = self.gat_1([X, A])
        Z = self.gat_2([H, A])

        return Z

In [ ]:
class InnerProductDecoder(Layer):
    """
    Reconstruct adjacency matrix from node embeddings.

    Return:
    A_hat: Reconstruct adjacency.
    """

    def call(self, Z):

        logits = tf.matmul(Z, Z, transpose_b=True)

        A_hat = tf.sigmoid(logits)

        return A_hat

In [94]:
class GraphAttentionAutoencoder(Model):
    """
    Full model:
    X, A -> GAT encoder -> Z -> decoder -> A_hat
    """

    def __init__(self, hidden_dim, embedding_dim):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim

        self.encoder = GATEncoder(
            hidden_dim=self.hidden_dim,
            embedding_dim=self.embedding_dim
        )

        self.decoder = InnerProductDecoder()

    def call(self, inputs):

        X, A = inputs

        Z = self.encoder([X, A])
        A_hat = self.decoder(Z)

        return A_hat

    def embed_nodes(self, X, A):
        """
        Return node-level embeddings.
        """
        return self.encoder([X, A])

    def embed_graphs(self, X, A):
        """
        Return graph-level embeddings using mean pooling.
        """

        Z = self.encoder([X, A])

        graph_embeddings = tf.reduce_mean(Z, axis=1)

        return graph_embeddings

In [95]:
def reconstruction_loss(A_true, A_pred):
    """
    Binary cross-entropy reconstruction loss.

    Input:
    A_true: original adjacency matrix
    A_pred: reconstructed adjacency matrix

    Output:
    scalar loss
    """

    A_binary = tf.cast(A_true > 0, tf.float32)

    loss = tf.keras.losses.binary_crossentropy(
        A_binary,
        A_pred
    )

    return tf.reduce_mean(loss)

In [96]:
def train_gate(
    model,
    optimizer,
    input_tensor,
    epochs=300
):
    """
    Train the graph attention autoencoder.
    """

    A_tensor = input_tensor[1]

    losses = []

    for epoch in range(epochs):

        with tf.GradientTape() as tape:

            A_pred = model(input_tensor)

            loss = reconstruction_loss(
                A_true=A_tensor,
                A_pred=A_pred
            )

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        losses.append(float(loss))

        if epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

    return losses

### GATE - Alternative Model with Graph Features

In [ ]:
class GATEncoderGraph(Layer):
    """
    Converts node-level embeddings into graph-level match embeddings.

    Inputs:
    Z_nodes:        (batch_size, N_nodes, node_embedding_dim)
    graph_features: (batch_size, n_graph_features)

    Output:
    graph_embedding: (batch_size, graph_embedding_dim)
    """

    def __init__(self, graph_embedding_dim, **kwargs):
        super().__init__(**kwargs)

        self.graph_embedding_dim = graph_embedding_dim

        # Linear activation
        self.graph_projection = Dense(
            graph_embedding_dim,
            activation="linear",
            name="graph_projection"
        )

    def call(self, inputs):

        Z_nodes, graph_features = inputs

        # Mean pooling over players/nodes
        pooled_node_embedding = tf.reduce_mean(
            Z_nodes,
            axis=1
        )

        # Concatenate learned node-based match embedding
        # with handcrafted graph-level metrics
        combined_graph_representation = tf.concat(
            [pooled_node_embedding, graph_features],
            axis=-1
        )

        # Final graph embedding
        graph_embedding = self.graph_projection(
            combined_graph_representation
        )

        return graph_embedding

In [98]:
class GATDecoderGraph(Layer):
    """
    Decodes the graph-level embedding into an adjacency-like tensor.

    Input:
    graph_embedding: (batch_size, graph_embedding_dim)

    Output:
    decoded_graph_tensor: (batch_size, N_nodes, N_nodes)
    """

    def __init__(self, n_nodes, **kwargs):
        super().__init__(**kwargs)

        self.n_nodes = n_nodes

        self.decoder_dense = Dense(
            n_nodes * n_nodes,
            activation="relu",
            name="graph_decoder_dense"
        )

        self.reshape_layer = Reshape(
            (n_nodes, n_nodes),
            name="graph_decoder_reshape"
        )

    def call(self, graph_embedding):

        decoded_flat = self.decoder_dense(graph_embedding)

        decoded_graph_tensor = self.reshape_layer(decoded_flat)

        return decoded_graph_tensor

In [ ]:
class GraphAttentionAutoencoderWithGraphFeatures(Model):
    """
    Graph Attention Autoencoder with node-level and graph-level features

    Inputs
    ------
    X_nodes: (batch_size, N_nodes, n_node_features)

    A: (batch_size, N_nodes, N_nodes)

    graph_features: (batch_size, n_graph_features)

    Output
    ------
    A_hat: (batch_size, N_nodes, N_nodes)
    """

    def __init__(self,n_nodes,hidden_dim,node_embedding_dim,graph_embedding_dim,**kwargs):
        super().__init__(**kwargs)

        self.n_nodes = n_nodes
        self.hidden_dim = hidden_dim
        self.node_embedding_dim = node_embedding_dim
        self.graph_embedding_dim = graph_embedding_dim

        # GAT encoder
        self.gat_encoder_node = GATEncoder(
            hidden_dim=self.hidden_dim,
            embedding_dim=self.node_embedding_dim
        )

        # Graph-level encoder
        self.gat_encoder_graph = GATEncoderGraph(
            graph_embedding_dim=self.graph_embedding_dim
        )

        # Graph-level decoder
        self.gat_decoder_graph = GATDecoderGraph(
            n_nodes=self.n_nodes
        )

        # Final decoder
        self.inner_product_decoder = InnerProductDecoder()

    def call(self, inputs):

        X_nodes, A, graph_features = inputs

        Z_nodes = self.gat_encoder_node([X_nodes, A])

        graph_embedding = self.gat_encoder_graph(
            [Z_nodes, graph_features]
        )

        decoded_graph_tensor = self.gat_decoder_graph(
            graph_embedding
        )

        A_hat = self.inner_product_decoder(
            decoded_graph_tensor
        )

        return A_hat

    def embed_nodes(self, inputs):
        """
        Return node embeddings.

        Output shape: (batch_size, N_nodes, node_embedding_dim)
        """

        X_nodes, A, _ = inputs

        Z_nodes = self.gat_encoder_node([X_nodes, A])

        return Z_nodes

    def embed_graphs(self, inputs):
        """
        Return graph embeddings.

        Output shape:(batch_size, graph_embedding_dim)
        """

        X_nodes, A, graph_features = inputs

        Z_nodes = self.gat_encoder_node([X_nodes, A])

        graph_embedding = self.gat_encoder_graph(
            [Z_nodes, graph_features]
        )

        return graph_embedding

#### Analysis of the clusters

In [100]:
def plot_tsne_kMean_graph_embedded_extend(G_embedded,label="Graph Clustering - Women Football 2023/24"):

    plt.figure(figsize=(11, 8))

    scatter = sns.scatterplot(
        data=G_embedded,
        x="tsne_1",
        y="tsne_2",
        hue="kMean_cluster",
        size="xg_team_match",
        style="competition",
        sizes=(80, 450),
        palette="Set2",
        alpha=0.8,
        edgecolor="black"
    )

    plt.title(
        label=label,
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel("t-SNE component 1")
    plt.ylabel("t-SNE component 2")

    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

In [101]:
#Triad Bar Plot

def create_triad_plot(G_embedded,label="Triangles Type Distribution by Cluster"):
    triad_cols = [
        c for c in G_embedded.columns
        if c.startswith("triad_")
    ]

    cluster_triads = (
        G_embedded
        .groupby("kMean_cluster")[triad_cols]
        .mean()
    )

    cluster_triads.T.plot(
        kind="bar",
        figsize=(15,6)
    )
    plt.title(label)

In [102]:
#Radar Plot

def create_radar_plot(G_embedded,label="Tactical Profiles of Passing Network Clusters"):
    metrics_cols = [
        "density",
        "average_degree",
        "degree_centralization",
        "average_clustering",
        "modularity",
        "average_path_length",
        "betweenness_sd",
        "algebraic_connectivity",
        "degree_assortativity",
        "entropy_motif",
        "score",
        "xg_team_match"
    ]

    cluster_profile = (
        G_embedded
        .groupby("kMean_cluster")[metrics_cols]
        .mean()
    )

    cluster_profile = cluster_profile.rename(
        columns={
            "density": "Density",
            "average_degree": "Average Degree",
            "degree_centralization": "Degree Centralization",
            "average_clustering": "Average Clustering",
            "modularity": "Modularity",
            "average_path_length": "Average Path Length",
            "betweenness_sd": "Betweenness SD",
            "algebraic_connectivity": "Algebraic Connectivity",
            "degree_assortativity": "Degree Assortativity",
            "entropy_motif": "Motif Entropy",
            "score": "Gol Score",
            "xg_team_match": "Match xG"
        }
    )

    scaler = StandardScaler()

    cluster_profile_z = pd.DataFrame(
        scaler.fit_transform(cluster_profile),
        index=cluster_profile.index,
        columns=cluster_profile.columns
    )



    fig = go.Figure()

    for cluster_id in cluster_profile_z.index:

        values = cluster_profile_z.loc[cluster_id].values

        values = list(values) + [values[0]]

        categories = (
            cluster_profile_z.columns.tolist()
            + [cluster_profile_z.columns[0]]
        )

        fig.add_trace(
            go.Scatterpolar(
                r=values,
                theta=categories,
                fill='toself',
                name=f'Cluster {cluster_id}'
            )
        )

    fig.update_layout(
        title=dict(
            text=label,
            x=0.5
        ),
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[-2, 2]
            )
        ),
        showlegend=True,
        width=900,
        height=700
    )

    fig.show()

In [103]:
#Radar Plot

def create_radar_plot2(G_embedded,label="Tactical Profiles of Passing Network Clusters"):
    metrics_cols = [
        "density",
        "average_degree",
        "degree_centralization",
        "average_clustering",
        "modularity",
        "average_path_length",
        "betweenness_sd",
        "algebraic_connectivity",
        "degree_assortativity",
        "entropy_motif",
        "score",
        "xg_team_match"
    ]


    scaler = MinMaxScaler()

    cluster_profile = pd.DataFrame(
        scaler.fit_transform(G_embedded[metrics_cols]),
        index=G_embedded[metrics_cols].index,
        columns=G_embedded[metrics_cols].columns
    )

    cluster_profile['kMean_cluster'] = G_embedded['kMean_cluster']

    cluster_profile_z = (
        cluster_profile
        .groupby("kMean_cluster")[metrics_cols]
        .mean()
    )


    cluster_profile_z = cluster_profile_z.rename(
        columns={
            "density": "Density",
            "average_degree": "Average Degree",
            "degree_centralization": "Degree Centralization",
            "average_clustering": "Average Clustering",
            "modularity": "Modularity",
            "average_path_length": "Average Path Length",
            "betweenness_sd": "Betweenness SD",
            "algebraic_connectivity": "Algebraic Connectivity",
            "degree_assortativity": "Degree Assortativity",
            "entropy_motif": "Motif Entropy",
            "score": "Gol Score",
            "xg_team_match": "Match xG"
        }
    )

    fig = go.Figure()

    for cluster_id in cluster_profile_z.index:

        values = cluster_profile_z.loc[cluster_id].values

        values = list(values) + [values[0]]

        categories = (
            cluster_profile_z.columns.tolist()
            + [cluster_profile_z.columns[0]]
        )

        fig.add_trace(
            go.Scatterpolar(
                r=values,
                theta=categories,
                fill='toself',
                name=f'Cluster {cluster_id}'
            )
        )

    fig.update_layout(
        title=dict(
            text=label,
            x=0.5
        ),
        polar=dict(
            radialaxis=dict(
                visible=Flase,
                range=[-0.5, 1]
            )
        ),
        showlegend=True,
        width=900,
        height=700
    )

    fig.show()

#### Cluster Choice

In [ ]:
def evaluate_kmeans_clusters(X, k_range=range(2, 11), random_state=42):
    """
    Evaluate KMeans clustering for different K values

    Metrics:
    - Silhouette
    - Davies-Bouldin Index
    - Calinski-Harabasz Index
    """

    results = []

    for k in k_range:

        kmeans = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=50
        )

        labels = kmeans.fit_predict(X)

        results.append({
            "K": k,
            "inertia": kmeans.inertia_,
            "silhouette_score": silhouette_score(X, labels),
            "davies_bouldin_index": davies_bouldin_score(X, labels),
            "calinski_harabasz_index": calinski_harabasz_score(X, labels)
        })

    return pd.DataFrame(results)

In [105]:
def plot_cluster_evaluation(cluster_eval):

    metrics = [
        "inertia",
        "silhouette_score",
        "davies_bouldin_index",
        "calinski_harabasz_index"
    ]

    for metric in metrics:

        plt.figure(figsize=(7, 5))

        plt.plot(
            cluster_eval["K"],
            cluster_eval[metric],
            marker="o"
        )

        plt.xlabel("Number of clusters K")
        plt.ylabel(metric)
        plt.title(f"KMeans evaluation: {metric}")
        plt.grid(alpha=0.3)

        plt.show()


In [ ]:
def cluster_stability_analysis(
    X,
    k_range=range(2, 11),
    seeds=range(20),
    n_init=20
):
    """
    Evaluate cluster stability across random seeds

    Uses Adjusted Rand Index between cluster assignments
    """

    stability_results = []

    for k in k_range:

        labels_by_seed = []

        for seed in seeds:

            kmeans = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=n_init
            )

            labels = kmeans.fit_predict(X)

            labels_by_seed.append(labels)

        ari_scores = []

        for i in range(len(labels_by_seed)):
            for j in range(i + 1, len(labels_by_seed)):

                ari = adjusted_rand_score(
                    labels_by_seed[i],
                    labels_by_seed[j]
                )

                ari_scores.append(ari)

        stability_results.append({
            "K": k,
            "mean_ARI": np.mean(ari_scores),
            "std_ARI": np.std(ari_scores),
            "min_ARI": np.min(ari_scores),
            "max_ARI": np.max(ari_scores)
        })

    return pd.DataFrame(stability_results)

In [ ]:
def internal_clustering_scores(X, labels):
    """
    Compute internal clustering metrics
    """

    X = np.asarray(X)
    labels = np.asarray(labels)

    n_clusters = len(np.unique(labels))

    if n_clusters < 2:
        raise ValueError("At least 2 clusters are required.")

    return {
        "silhouette": silhouette_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels)
    }

In [ ]:
def compactness_separation(X, labels):
    """
    Compute intra-cluster compactness and inter-cluster separation
    """

    X = np.asarray(X)
    labels = np.asarray(labels)

    unique_labels = np.unique(labels)

    centroids = []
    intra_distances = []

    for lab in unique_labels:
        cluster_points = X[labels == lab]
        centroid = cluster_points.mean(axis=0)
        centroids.append(centroid)

        distances = np.linalg.norm(cluster_points - centroid, axis=1)
        intra_distances.append(np.mean(distances))

    centroids = np.vstack(centroids)

    centroid_distances = cdist(centroids, centroids)

    inter_distances = centroid_distances[np.triu_indices_from(
        centroid_distances,
        k=1
    )]

    return {
        "intra_cluster_distance": np.mean(intra_distances),
        "inter_cluster_distance": np.mean(inter_distances),
        "separation_ratio": np.mean(inter_distances) / np.mean(intra_distances)
    }

In [109]:
def evaluate_embedding_method(
    name,
    X,
    labels
):
    """
    Evaluate one embedding method.
    """

    results = {
        "embedding": name
    }

    results.update(internal_clustering_scores(X, labels))
    results.update(compactness_separation(X, labels))

    return results

#### GAT for prediction of xG

In [ ]:
#train test split

def split_graph_dataset(
    X_all,
    A_all,
    graph_features_all,
    target,
    test_size=0.15,
    val_size=0.15,
    random_state=42
):
    """
    Split graph tensors into train, validation, and test sets

    Inputs
    ------
    X_all: (num_graphs, max_nodes, node_features)

    A_all: (num_graphs, max_nodes, max_nodes)

    graph_features_all: (num_graphs, graph_features)

    target (xG): (num_graphs, 1)

    Outputs
    -------
    Final split train/val/test tensors
    """

    graph_indices = np.arange(len(X_all))

    train_val_idx, test_idx = train_test_split(
        graph_indices,
        test_size=test_size,
        random_state=random_state,
        shuffle=True
    )

    relative_val_size = val_size / (1 - test_size)

    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=relative_val_size,
        random_state=random_state,
        shuffle=True
    )

    dataset_split = {
        "X_train": X_all[train_idx],
        "A_train": A_all[train_idx],
        "G_train": graph_features_all[train_idx],
        "target_train": target.iloc[train_idx].reset_index(drop=True),

        "X_val": X_all[val_idx],
        "A_val": A_all[val_idx],
        "G_val": graph_features_all[val_idx],
        "target_val": target.iloc[val_idx].reset_index(drop=True),

        "X_test": X_all[test_idx],
        "A_test": A_all[test_idx],
        "G_test": graph_features_all[test_idx],
        "target_test": target.iloc[test_idx].reset_index(drop=True),
    }

    return dataset_split

In [ ]:
#create tf database
def create_xg_tf(
    X,
    A,
    graph_features,
    targets,
    GAT_Adjust = False, 
    batch_size=32,
    shuffle=True,
    buffer_size=1000
):
    """
    Create a batched TensorFlow dataset for graph autoencoder training
    """

    X = X.astype(np.float32)
    A = A.astype(np.float32)
    graph_features = graph_features.astype(np.float32)

    if GAT_Adjust:
        inputs = (
            X,
            A,
            graph_features
        )
    else:
        inputs = (
            X,
            A
        )

    targets = np.asarray(targets)
    targets = np.log1p(targets)
    targets = targets.reshape(-1,1).astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (inputs, targets)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(buffer_size, len(X)),
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [ ]:
#create the 3 tf 

def prepare_dataloaders(
    dataset_split,
    GAT_Adjust = False,
    batch_size=32
):
    """
    Create train, validation, and test dataloaders
    """

    train_dataset = create_xg_tf(
        X=dataset_split["X_train"],
        A=dataset_split["A_train"],
        graph_features=dataset_split["G_train"],
        targets=dataset_split["target_train"],
        GAT_Adjust = GAT_Adjust,
        batch_size=batch_size,
        shuffle=True
    )

    val_dataset = create_xg_tf(
        X=dataset_split["X_val"],
        A=dataset_split["A_val"],
        graph_features=dataset_split["G_val"],
        targets=dataset_split["target_val"],
        GAT_Adjust = GAT_Adjust,
        batch_size=batch_size,
        shuffle=False
    )

    test_dataset = create_xg_tf(
        X=dataset_split["X_test"],
        A=dataset_split["A_test"],
        graph_features=dataset_split["G_test"],
        targets=dataset_split["target_test"],
        GAT_Adjust = GAT_Adjust,
        batch_size=batch_size,
        shuffle=False
    )

    return train_dataset, val_dataset, test_dataset

In [ ]:
class GraphAttentionxGRegression(Model):

    """
    Graph Attention Network for graph-level xG prediction.

    Inputs
    ------
    X_nodes: (batch_size, N_nodes, n_node_features)

    A: (batch_size, N_nodes, N_nodes)

    Output
    ------
    y_pred: (batch_size, 1)
    """

    def __init__(
        self,
        hidden_dim: int = 64,
        embedding_dim: int = 32,
        dense_dim: int = 64,
        dropout_rate: float = 0.20,
        name: str = "graph_attention_xg_regressor"
    ):
        super().__init__(name=name)


        self.encoder = GATEncoder(hidden_dim=hidden_dim, embedding_dim=embedding_dim)

        self.dropout = Dropout(dropout_rate)

        self.dense_hidden = Dense(dense_dim,activation='relu',name="dense_hidden")

        self.output_layer = Dense(1,activation='linear',name='xg_output')
    
    def call(self,inputs,training='False'):

        X_all, A_all = inputs


        Z_nodes = self.encoder([X_all, A_all])

        # Mean pooling: average player representation

        graph_mean = tf.reduce_mean(Z_nodes, axis=1)

        # Max pooling: strongest player-level signal

        graph_max = tf.reduce_max(Z_nodes, axis=1)

        # Concatenate both graph summaries

        graph_embedding = tf.concat(
            [graph_mean, graph_max],
            axis=-1
        )

        graph_embedding = self.dropout(
            graph_embedding,
            training=training
        )

        hidden = self.dense_hidden(graph_embedding)

        output = self.output_layer(hidden)

        return output

In [ ]:
def predict_xg(model, dataset, predict_log_xg=True):
    """
    Predict xG from a trained model.
    """

    y_true_all = []
    y_pred_all = []

    for (X_batch, A_batch), y_batch in dataset:

        y_pred = model(
            [X_batch, A_batch],
            training=False
        )

        y_true_all.append(y_batch.numpy())
        y_pred_all.append(y_pred.numpy())

    y_true = np.vstack(y_true_all).flatten()
    y_pred = np.vstack(y_pred_all).flatten()

    y_true = np.expm1(y_true)
    y_pred = np.expm1(y_pred)

    y_pred = np.maximum(y_pred, 0)

    return y_true, y_pred

In [115]:
def regression_report(y_true, y_pred):
    """
    Compute regression metrics on original xG scale.
    """

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    report = pd.DataFrame({
        "metric": ["MAE", "MSE", "RMSE", "R2"],
        "value": [mae, mse, rmse, r2]
    })

    return report

In [116]:
def plot_training_history(history):
    """
    Plot training and validation loss curves.
    """

    history_dict = history.history if hasattr(history, "history") else history

    plt.figure(figsize=(8, 5))

    plt.plot(
        history_dict["loss"] if "loss" in history_dict else history_dict["train_loss"],
        label="Train loss"
    )

    plt.plot(
        history_dict["val_loss"],
        label="Validation loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("GAT xG Regression Training Curve")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.show()

In [117]:
def plot_predicted_vs_true(y_true, y_pred, title="Predicted vs True xG"):

    plt.figure(figsize=(6, 6))

    plt.scatter(
        y_true,
        y_pred,
        alpha=0.7,
        edgecolors="black"
    )

    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())

    plt.plot(
        [min_value, max_value],
        [min_value, max_value],
        linestyle="--"
    )

    plt.xlabel("True xG")
    plt.ylabel("Predicted xG")
    plt.title(title)
    plt.grid(alpha=0.3)

    plt.show()

### Final Results Summary

In [ ]:
def summarize_features_by_gender(
    features_df,
    gender_column= "competition_gender",
    feature_columns = 12,
    round_digits = 2
) :
    """
    Compute descriptive statistics for each numerical graph feature, separately for male and female competitions

    Parameters
    ----------
    features_df : DataFrame containing one row per match-team graph, numerical graph features

    gender_column 

    exclude_columns Numerical columns that should not be treated as graph features


    Returns
    -------
    summary table 
    """

    summary_rows = []

    # Calculate the statistics independently for each gender group.
    for gender_value, gender_df in features_df.groupby(
        gender_column,
        dropna=False,
        observed=True
    ):

        for feature_name in feature_columns:

            feature_values = (
                gender_df[feature_name]
                .replace([np.inf, -np.inf], np.nan)
            )

            valid_values = feature_values.dropna()

            n_total = len(feature_values)

            summary_rows.append({
                "feature": feature_name,
                "competition_gender": gender_value,
                "n_total": n_total,
                "mean": valid_values.mean(),
                "std": valid_values.std(ddof=1),
                "min": valid_values.min(),
                "median": valid_values.median(),
                "max": valid_values.max()
            })

    summary_df = (
        pd.DataFrame(summary_rows)
        #.sort_values(["competition_gender"])
        .reset_index(drop=True)
    )

    return summary_df.round(round_digits)

In [ ]:
def create_gender_comparison_table(summary_df):

    required_columns = {
        "feature",
        "competition_gender",
        "mean",
        "std",
        "median",
        "min",
        "max"
    }

    missing_columns = required_columns.difference(summary_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    df = summary_df.copy()

    # Standardise the gender labels
    df["competition_gender"] = (
        df["competition_gender"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Reshape from long format to wide format
    comparison = df.pivot(
        index="feature",
        columns="competition_gender",
        values=["mean",
        "std",
        "median",
        "min",
        "max"]
    )

    # Flatten the multi-level column names
    comparison.columns = [
        f"{gender}_{stat}".lower()
        for stat, gender in comparison.columns
    ]

    comparison = comparison.reset_index()

    # Rename columns using dissertation-friendly labels
    comparison = comparison.rename(
        columns={
            "feature": "Feature",
            "female_mean": "Female Mean",
            "female_std": "Female SD",
            "female_median": "Female Median",
            "female_min": "Female Min",
            "female_max": "Female Max",
            "male_mean": "Male Mean",
            "male_std": "Male SD",
            "male_median": "Male Median",
            "male_min": "Male Min",
            "male_max": "Male Max"
        }
    )

    # Select the required column order
    column_order = [
        "Feature",
        "Female Mean",
        "Female SD",
        "Female Median",
        "Female Min",
        "Female Max",
        "Male Mean",
        "Male SD",
        "Male Median",
        "Male Min",
        "Male Max"
    ]

    comparison = comparison[column_order]

    return comparison

In [ ]:
def create_compact_gender_table(comparison_df,decimals= 2) :


    df = comparison_df.copy()

    df["Female Mean ± SD"] = (
        df["Female Mean"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
        + " ± "
        + df["Female SD"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
    )

    df["Male Mean ± SD"] = (
        df["Male Mean"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
        + " ± "
        + df["Male SD"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
    )

    compact_table = df[
        [
            "Feature",
            "Female Mean ± SD",
            "Male Mean ± SD"
        ]
    ].copy()

    return compact_table

In [ ]:
def create_cluster_comparison_table(summary_df) :

    required_columns = {
        "feature",
        "competition_gender",
        "mean",
        "std",
        "median",
        "min",
        "max"
    }

    missing_columns = required_columns.difference(summary_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    df = summary_df.copy()

    # Standardise the gender labels
    df["competition_gender"] = (
        df["competition_gender"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Reshape from long format to wide format
    comparison = df.pivot(
        index="feature",
        columns="competition_gender",
        values=["mean",
        "std",
        "median",
        "min",
        "max"]
    )

    # Flatten the multi-level column names
    comparison.columns = [
        f"{gender}_{stat}".lower()
        for stat, gender in comparison.columns
    ]

    comparison = comparison.reset_index()

    return comparison

In [ ]:
def create_compact_cluster_table(comparison_df,decimals = 2):


    df = comparison_df.copy()

    df["Cluster 0 Mean ± SD"] = (
        df["0_mean"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
        + " ± "
        + df["0_std"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
    )

    df["Cluster 1 Mean ± SD"] = (
        df["1_mean"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
        + " ± "
        + df["1_std"].round(decimals).map(
            lambda value: f"{value:.{decimals}f}"
        )
    )

    compact_table = df[
        [
            "feature",
            "Cluster 0 Mean ± SD",
            "Cluster 1 Mean ± SD"
        ]
    ].copy()

    return compact_table

### SBM

In [ ]:
def dc_sbm_log_likelihood_extended(G,partition,weight = "pass_freq"):
    """
    Compute the degree-corrected SBM profile log-likelihood.

    """
    blocks = sorted(set(partition.values()))

    # Total weighted degree in every block
    kappa = {block: 0.0 for block in blocks}

    for node in G.nodes():
        block = partition[node]

        kappa[block] += G.degree(
            node,
            weight=weight
        )

    # Weighted block-to-block matrix
    block_matrix = {
        (block_r, block_s): 0.0
        for block_r in blocks
        for block_s in blocks
    }

    for source, target, edge_data in G.edges(data=True):

        block_r = partition[source]
        block_s = partition[target]

        edge_weight = float(
            edge_data.get(weight, 1.0)
        )

        block_matrix[(block_r, block_s)] += edge_weight

        # Add the symmetric entry only for undirected graphs
        if not G.is_directed():
            block_matrix[(block_s, block_r)] += edge_weight

    log_likelihood = 0.0

    for block_r in blocks:
        for block_s in blocks:

            m_rs = block_matrix[(block_r, block_s)]

            denominator = (
                kappa[block_r] *
                kappa[block_s]
            )

            if m_rs > 0 and denominator > 0:
                log_likelihood += (
                    m_rs *
                    np.log(m_rs / denominator)
                )

    return float(log_likelihood), block_matrix

In [ ]:
def fit_dc_sbm_from_partition_extended(
    G,
    weight="pass_freq",
    max_iter=100,
    random_state=42,
    relabel_final_blocks=True
):
    """
    Fit a degree-corrected SBM using greedy node reassignment

    The initial partition is obtained using Girvan-Newman community detection
 
    Returns
    -------
    results : Dictionary containing the final partition, initial and iterations
    """

    if G.number_of_nodes() == 0:
        raise ValueError("The graph contains no nodes.")

    random.seed(random_state)
    np.random.seed(random_state)


    # 1. Initial Girvan-Newman partition

    initial_communities = find_community_detection(G)

    initial_partition = partition_from_communities(
        initial_communities
    )

    partition = initial_partition.copy()

    initial_blocks = sorted(set(partition.values()))
    initial_K = len(initial_blocks)

    # These are the candidate labels available during optimisation.
    candidate_blocks = initial_blocks.copy()

    initial_L = dc_sbm_log_likelihood(
        G,
        partition,
        weight=weight
    )

    current_L = initial_L

    converged = False
    iterations_completed = 0
    accepted_moves = 0

    # 2. reassignment

    for iteration_index in range(max_iter):

        improved = False
        nodes = list(G.nodes())
        random.shuffle(nodes)

        for node in nodes:

            current_block = partition[node]
            best_block = current_block
            best_L = current_L

            for new_block in candidate_blocks:

                if new_block == current_block:
                    continue

                test_partition = partition.copy()
                test_partition[node] = new_block

                test_L = dc_sbm_log_likelihood(
                    G,
                    test_partition,
                    weight=weight
                )

                if test_L > best_L:
                    best_L = test_L
                    best_block = new_block

            if best_block != current_block:
                partition[node] = best_block
                current_L = best_L
                improved = True
                accepted_moves += 1

        iterations_completed = iteration_index + 1

        if not improved:
            converged = True
            break

    # 3. Determine non-empty final blocks

    final_block_sizes_original = dict(
        Counter(partition.values())
    )

    non_empty_final_blocks = sorted(
        final_block_sizes_original.keys()
    )

    final_K = len(non_empty_final_blocks)

    empty_blocks = sorted(
        set(candidate_blocks)
        - set(non_empty_final_blocks)
    )


    # 4. Optionally relabel remaining blocks consecutively

    if relabel_final_blocks:

        block_mapping = {
            old_block: new_block
            for new_block, old_block
            in enumerate(non_empty_final_blocks)
        }

        final_partition = {
            node: block_mapping[old_block]
            for node, old_block in partition.items()
        }

    else:
        block_mapping = {
            block: block
            for block in non_empty_final_blocks
        }

        final_partition = partition.copy()

    final_block_sizes = dict(
        Counter(final_partition.values())
    )

    # Relabelling does not change the likelihood, but recomputing it
    # provides a final consistency check.
    final_L = dc_sbm_log_likelihood(
        G,
        final_partition,
        weight=weight
    )

    return final_partition, final_K, iterations_completed


In [125]:
def dc_sbm_gn_results_extended(G,weight="pass_freq"):

    initial_partition = find_community_detection(G)

    initial_partition = partition_from_communities(initial_partition)

    initial_L = dc_sbm_log_likelihood(G, initial_partition, weight=weight)

    dc_partition_gn, K, iteration = fit_dc_sbm_from_partition_extended(G)
    
    final_L = dc_sbm_log_likelihood(G,dc_partition_gn, weight=weight)

    likelihood_improvement = final_L - initial_L

    total_passes = sum(
        float(data.get(weight, 1.0))
        for _, _, data in G.edges(data=True)
    )

    normalised_final_log_likelihood = (
        final_L / total_passes
        if total_passes > 0
        else np.nan
    )

    dc_sbm_gn_results = pd.DataFrame({
        "player": list(dc_partition_gn.keys()),
        "block": list(dc_partition_gn.values())
    }).sort_values("block")

    dc_partition = []

    number_of_final_blocks = len(
        set(dc_partition_gn.values())
    )

    number_of_initial_blocks = len(
        set(initial_partition.values())
    )

    for i in range(K):
        dc_partition.append(list(dc_sbm_gn_results[dc_sbm_gn_results['block'] ==i]['player'].values))
    
    return {
        "initial_partition": initial_partition,
        "final_partition": dc_partition,
        "initial_log_likelihood": initial_L,
        "final_log_likelihood": final_L,
        "likelihood_improvement": likelihood_improvement,
        "normalised_final_log_likelihood": normalised_final_log_likelihood,
        "number_of_initial_blocks": number_of_initial_blocks,
        "number_of_final_blocks": number_of_final_blocks,
        "total_passes": total_passes,
        "number_iteration": iteration
    }

In [126]:
def build_dc_sbm_features(matches,
                          G_gen,
                          weight = "pass_freq"):
    all_metrics = []

    for match_id in matches["match_id"].values:

        competition = matches[(matches['match_id'] == match_id)]['competition']
        score = matches[(matches['match_id'] == match_id)]['home_score'] # capture the home score
        home_dummy = 1
        competition_gender = matches[(matches['match_id'] == match_id)]['competition_gender']

        events = sb.events(match_id=match_id)
        
        for team_id in events['team_id'].unique():

            passes = create_passes_db(events, team_id)
            G_match = G_gen(passes)
            dict_sbm = dc_sbm_gn_results_extended(G_match,weight=weight)
            metric_row = {
                "match_id": match_id,
                "team_id": team_id,
                "team_name": passes["team"].values[0],
                "competition": competition.values[0],
                "competition_gender": competition_gender.values[0],
                "home_dummy": home_dummy,
                "score": score.values[0],
                "xg_team_match": events[(events['team_id'] == team_id)]['shot_statsbomb_xg'].dropna().sum(),
                "number_of_players": G_match.number_of_nodes(),
                "initial_log_likelihood": (
                    dict_sbm["initial_log_likelihood"]
                ),
                "final_log_likelihood": (
                    dict_sbm["final_log_likelihood"]
                ),
                "likelihood_improvement": (
                    dict_sbm["likelihood_improvement"]
                ),
                "normalised_final_log_likelihood": (
                    dict_sbm[
                        "normalised_final_log_likelihood"
                    ]
                ),
                "number_of_final_blocks": (
                    dict_sbm["number_of_final_blocks"]
                ),
                "total_passes": dict_sbm["total_passes"],
                "number_iteration": (
                    dict_sbm["number_iteration"]
                )
            }

            score = matches[(matches['match_id'] == match_id)]['away_score'] # to capture the away score
            home_dummy = 0
            all_metrics.append(metric_row)

    metrics_df = pd.DataFrame(all_metrics)

    return metrics_df

In [ ]:
def significance_stars(p_value):
    """
    Return significance stars based on the p-value.

    ----------
    *** : p < 0.001
    **  : p < 0.01
    *   : p < 0.05
    no star : p >= 0.05
    """

    if pd.isna(p_value):
        return ""

    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    else:
        return ""


def create_sbm_gender_table(
    df,
    gender_column = "competition_gender",
    block_column = "number_of_final_blocks",
    players_per_block_column= "players_block",
    xg_column = "xg_team_match",
    round_digits = 3
) :
    """
    Create a male-female comparison table for SBM results

    """

    required_columns = {
        gender_column,
        block_column,
        players_per_block_column,
        xg_column
    }

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise KeyError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    analysis_df = df[
        [
            gender_column,
            block_column,
            players_per_block_column,
            xg_column
        ]
    ].copy()

    # Standardise gender labels
    analysis_df[gender_column] = (
        analysis_df[gender_column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Replace infinite values with missing values
    numeric_columns = [
        block_column,
        players_per_block_column,
        xg_column
    ]

    analysis_df[numeric_columns] = (
        analysis_df[numeric_columns]
        .replace([np.inf, -np.inf], np.nan)
    )

    results = {}
    correlation_rows = []

    for gender in ["male", "female"]:

        gender_df = analysis_df.loc[
            analysis_df[gender_column] == gender
        ].copy()

        average_blocks = gender_df[block_column].mean()

        average_players_per_block = (
            gender_df[players_per_block_column].mean()
        )

        # Keep only rows valid for both variables
        correlation_df = gender_df[
            [block_column, xg_column]
        ].dropna()

        correlation_df = correlation_df.loc[
            correlation_df[xg_column] >= 0
        ].copy()

        correlation_df["log_xg"] = np.log1p(
            correlation_df[xg_column]
        )

        # Pearson correlation requires variation in both variables
        if (
            len(correlation_df) >= 3
            and correlation_df[block_column].nunique() > 1
            and correlation_df["log_xg"].nunique() > 1
        ):
            correlation, p_value = pearsonr(
                correlation_df[block_column],
                correlation_df["log_xg"]
            )
        else:
            correlation = np.nan
            p_value = np.nan

        stars = significance_stars(p_value)

        results[gender.capitalize()] = {
            "Avg. number of non-empty final blocks": average_blocks,
            "Avg. players per non-empty final block": (
                average_players_per_block
            ),
            "Pearson correlation #block and log(1 + xG)": (
                f"{correlation:.{round_digits}f}{stars}"
                if not pd.isna(correlation)
                else "NA"
            )
        }

        correlation_rows.append({
            "competition_gender": gender,
            "N": len(correlation_df),
            "pearson_r": correlation,
            "p_value": p_value,
            "significance": stars
        })

    formatted_table = pd.DataFrame(results)


    for column in ["Male", "Female"]:
        formatted_table.loc[
            "Avg. number of non-empty final blocks",
            column
        ] = round(
            formatted_table.loc[
                "Avg. number of non-empty final blocks",
                column
            ],
            round_digits
        )

        formatted_table.loc[
            "Avg. players per non-empty final block",
            column
        ] = round(
            formatted_table.loc[
                "Avg. players per non-empty final block",
                column
            ],
            round_digits
        )

    correlation_details = (
        pd.DataFrame(correlation_rows)
        .round({
            "pearson_r": round_digits,
            "p_value": 4
        })
    )

    return formatted_table, correlation_details